# MCP Advanced Cache and Search Tool Demonstration

This notebook demonstrates the advanced MCP features added for local Perseus/Scaife research. It is intentionally written as a runnable quality check: each section explains the feature, calls the tool through the same FastMCP interface used by external clients, and includes small assertions that fail loudly if the response shape is not what the tool contract expects.

The workflow covers:

- local metadata cache inspection and refresh for faster repeated discovery;
- paged JSON citation references and lightweight reference counts;
- Scaife library search pagination plus server-side `text_group` and `work` scopes;
- author-scoped search, including the server-side textgroup path;
- lemma search and operator-preserving query syntax;
- reader search within one edition;
- passage-level token highlights;
- Scaife metadata, passage JSON, and passage plaintext retrieval for URNs that may not map cleanly to Perseus CTS.

> Requirements: run from the repository root or keep the path setup cell unchanged, install project dependencies, and have internet access to Perseus/Scaife upstream services. Counts and first-result ordering may change when upstream data changes.


## Setup

The setup cell imports the local `server.py`, reloads it so code changes are visible in an existing kernel, and defines helpers for calling MCP tools. It also pins `PERSEUS_MCP_CACHE_DIR` to the repository-level `.cache/perseus-mcp` directory before importing `server`.

That cache setting matters when a notebook is launched from `examples/`: without it, the default cache would be relative to the notebook process current working directory (`examples/.cache/perseus-mcp`). This is not a second MCP server instance; it is only a different cache location for a separate Python process. Pinning the environment variable keeps notebook and repo-root runs pointed at the same disk cache.


In [12]:
from pathlib import Path
import importlib
import json
import os
import sys

print("Loading notebook setup...")

START = Path.cwd().resolve()
REPO_ROOT = START
for candidate in [START, *START.parents]:
    if (candidate / "server.py").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError(f"Could not find server.py from {START}")

print(f"Repository root: {REPO_ROOT}")
sys.path.insert(0, str(REPO_ROOT))

# Keep notebook cache files at the repository root even when Jupyter starts in examples/.
os.environ.setdefault("PERSEUS_MCP_CACHE_DIR", str(REPO_ROOT / ".cache" / "perseus-mcp"))
print(f"Cache directory: {os.environ['PERSEUS_MCP_CACHE_DIR']}")

print("Importing FastMCP client and local server module...")
from fastmcp import Client
import server

server = importlib.reload(server)
mcp = server.mcp
print("MCP server module loaded.")


def tool_text(result):
    return "\n".join(
        block.text for block in result.content if getattr(block, "text", None) is not None
    )


async def call_json(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return json.loads(tool_text(result))


async def call_text(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return tool_text(result)


def summarize_search(data):
    results = data.get("results", [])
    first = results[0] if results else {}
    passage = first.get("passage", {})
    text = passage.get("text", {})
    snippet = " ".join(first.get("content", [])) if first else None
    return {
        "total_count": data.get("total_count"),
        "page": data.get("page", {}).get("number"),
        "num_pages": data.get("page", {}).get("num_pages"),
        "first_urn": passage.get("urn"),
        "first_text_label": text.get("label"),
        "first_snippet": snippet,
        "author_scope": data.get("author_scope"),
    }


Loading notebook setup...
Repository root: D:\Onedrive\GitHub\Perseus-mcp
Cache directory: D:\Onedrive\GitHub\Perseus-mcp\.cache\perseus-mcp
Importing FastMCP client and local server module...
MCP server module loaded.


## Tool Surface Check

Before testing behavior, confirm that the new advanced tools are actually registered with FastMCP. This catches stale kernels, wrong working directories, and older installed versions before any live network calls are made.


In [13]:
expected_new_tools = {
    "get_cache_status",
    "refresh_metadata_cache",
    "clear_metadata_cache",
    "get_valid_references_json",
    "count_valid_references",
    "search_within_text",
    "get_passage_highlights",
    "get_scaife_library_metadata",
    "get_scaife_passage_json",
    "get_scaife_passage_text",
}

async with Client(mcp) as client:
    tools = await client.list_tools()

tool_names = {tool.name for tool in tools}
missing = expected_new_tools - tool_names
assert not missing, f"Missing expected tools: {sorted(missing)}"

print(f"Registered tools: {len(tool_names)}")
print("New tools present:")
for name in sorted(expected_new_tools):
    print("-", name)


Registered tools: 23
New tools present:
- clear_metadata_cache
- count_valid_references
- get_cache_status
- get_passage_highlights
- get_scaife_library_metadata
- get_scaife_passage_json
- get_scaife_passage_text
- get_valid_references_json
- refresh_metadata_cache
- search_within_text


## Local Metadata Cache

`GetCapabilities` and large `GetValidReff` responses can be multi-megabyte requests. The server now caches this stable metadata in memory and on disk, which makes repeated local discovery and navigation much faster.

This section inspects the cache, refreshes CTS capabilities, and inspects the cache again. The guarded clear-cache cell is present for manual testing, but it stays disabled by default so running the notebook does not erase useful local metadata.


In [14]:
async with Client(mcp) as client:
    before = await call_json(client, "get_cache_status")
    refreshed = await call_json(client, "refresh_metadata_cache")
    after = await call_json(client, "get_cache_status")

assert "cache_dir" in after
assert after["disk_files"] >= before["disk_files"]

print("Before:")
print(json.dumps(before, ensure_ascii=False, indent=2))
print("\nRefresh result:")
print(json.dumps(refreshed, ensure_ascii=False, indent=2)[:1000])
print("\nAfter:")
print(json.dumps(after, ensure_ascii=False, indent=2))


Before:
{
  "enabled": true,
  "cache_dir": "D:\\Onedrive\\GitHub\\Perseus-mcp\\.cache\\perseus-mcp",
  "ttl_seconds": 86400,
  "memory_entries": 0,
  "disk_files": 2,
  "disk_bytes": 3139312
}

Refresh result:
{
  "refreshed": true,
  "cache": {
    "enabled": true,
    "cache_dir": "D:\\Onedrive\\GitHub\\Perseus-mcp\\.cache\\perseus-mcp",
    "ttl_seconds": 86400,
    "memory_entries": 1,
    "disk_files": 2,
    "disk_bytes": 3139312
  },
  "capabilities_bytes": 2052315
}

After:
{
  "enabled": true,
  "cache_dir": "D:\\Onedrive\\GitHub\\Perseus-mcp\\.cache\\perseus-mcp",
  "ttl_seconds": 86400,
  "memory_entries": 1,
  "disk_files": 2,
  "disk_bytes": 3139312
}


In [15]:
RUN_CACHE_CLEAR_DEMO = False

if RUN_CACHE_CLEAR_DEMO:
    async with Client(mcp) as client:
        cleared = await call_json(client, "clear_metadata_cache")
    print(json.dumps(cleared, ensure_ascii=False, indent=2))
else:
    print("Skipping clear_metadata_cache demo. Set RUN_CACHE_CLEAR_DEMO = True to run it.")


Skipping clear_metadata_cache demo. Set RUN_CACHE_CLEAR_DEMO = True to run it.


## Paged Citation References

Raw `GetValidReff` XML can be large. `count_valid_references` gives a quick size estimate, while `get_valid_references_json` returns a small paged slice with `limit`, `offset`, and `has_next`. This is the better default for agents that need to inspect or page through citations without loading an entire work into context.


In [16]:
ILIAD_CTS_EDITION = "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1"

async with Client(mcp) as client:
    ref_count = await call_json(
        client,
        "count_valid_references",
        {"urn": ILIAD_CTS_EDITION, "level": 1},
    )
    ref_page = await call_json(
        client,
        "get_valid_references_json",
        {"urn": ILIAD_CTS_EDITION, "level": 1, "limit": 5, "offset": 0},
    )

assert ref_count["total_count"] >= ref_page["returned_count"]
assert ref_page["returned_count"] <= 5

print(json.dumps(ref_count, ensure_ascii=False, indent=2))
print(json.dumps(ref_page, ensure_ascii=False, indent=2))


{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "level": 1,
  "total_count": 14956
}
{
  "urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
  "level": 1,
  "total_count": 14956,
  "offset": 0,
  "limit": 5,
  "returned_count": 5,
  "has_next": true,
  "references": [
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.2",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.3",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.4",
    "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.5"
  ]
}


## Server-Scoped Library Search

`search_perseus` now forwards `page_num`, `text_group`, `work`, and `result_format` to Scaife. Using Scaife server-side scopes is more accurate and efficient than fetching a broad page and filtering it locally.

The example searches for `μῆνιν` only within Homer's *Iliad* by passing both the Homer textgroup URN and the Iliad work URN.


In [17]:
async with Client(mcp) as client:
    scoped_search = await call_json(
        client,
        "search_perseus",
        {
            "query": "μῆνιν",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "form",
            "page_num": 1,
            "text_group": "urn:cts:greekLit:tlg0012",
            "work": "urn:cts:greekLit:tlg0012.tlg001",
            "result_format": "instances",
        },
    )

summary = summarize_search(scoped_search)
assert scoped_search["total_count"] >= len(scoped_search.get("results", []))
assert summary["first_urn"] is None or "tlg0012.tlg001" in summary["first_urn"]

print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "total_count": 9,
  "page": 1,
  "num_pages": 1,
  "first_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.75",
  "first_text_label": "Ἰλιάς",
  "first_snippet": "<em>μῆνιν</em> Ἀπόλλωνος ἑκατηβελέταο ἄνακτος·",
  "author_scope": null
}


## Author-Scoped Search

The `author` convenience option still accepts a human-friendly name such as `Homer`. Internally, the server resolves it against cached CTS capabilities. If the result is a single textgroup, that textgroup is sent to Scaife as a server-side filter; ambiguous author matches fall back to local URN-prefix filtering.


In [18]:
async with Client(mcp) as client:
    homer_search = await call_json(
        client,
        "search_perseus",
        {
            "query": '"μῆνιν ἄειδε"',
            "language": "greek",
            "query_format": "unicode",
            "preserve_operators": True,
            "author": "Homer",
        },
    )

assert "author_scope" in homer_search
print(json.dumps(summarize_search(homer_search), ensure_ascii=False, indent=2))


{
  "total_count": 43,
  "page": 1,
  "num_pages": 5,
  "first_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
  "first_text_label": "Ἰλιάς",
  "first_snippet": "<em>μῆνιν ἄειδε</em> θεὰ Πηληϊάδεω Ἀχιλῆος",
  "author_scope": {
    "query": "Homer",
    "match_count": 2,
    "urns": [
      "urn:cts:greekLit:tlg0012",
      "urn:cts:greekLit:tlg0012.tlg001",
      "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1",
      "urn:cts:greekLit:tlg0012.tlg001.perseus-eng1",
      "urn:cts:greekLit:tlg0012.tlg001.perseus-eng2",
      "urn:cts:greekLit:tlg0012.tlg002",
      "urn:cts:greekLit:tlg0012.tlg002.perseus-grc1",
      "urn:cts:greekLit:tlg0012.tlg002.perseus-eng1",
      "urn:cts:greekLit:tlg0012.tlg002.perseus-eng2",
      "urn:cts:greekLit:tlg0013",
      "urn:cts:greekLit:tlg0013.tlg027",
      "urn:cts:greekLit:tlg0013.tlg027.perseus-grc1",
      "urn:cts:greekLit:tlg0013.tlg027.perseus-eng1",
      "urn:cts:greekLit:tlg0013.tlg026",
      "urn:cts:greekLit:tlg0013.tlg026.pe

## Lemma and Operator-Preserving Search

`search_kind=lemma` asks Scaife to search lemmas instead of surface forms. `preserve_operators=True` keeps query syntax such as quotes, `-`, `|`, `*`, and `~` intact. This matters because some of those characters also have Beta Code meanings.

The example uses an OR-style lemma query, `λόγος | ἀνήρ`, and forces Unicode handling so the operator reaches Scaife unchanged.


In [19]:
async with Client(mcp) as client:
    lemma_or_search = await call_json(
        client,
        "search_perseus",
        {
            "query": "λόγος | ἀνήρ",
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "lemma",
            "preserve_operators": True,
        },
    )

assert lemma_or_search["total_count"] >= len(lemma_or_search.get("results", []))
print(json.dumps(summarize_search(lemma_or_search), ensure_ascii=False, indent=2))


{
  "total_count": 20086,
  "page": 1,
  "num_pages": 2009,
  "first_urn": "urn:cts:greekLit:tlg0085.tlg005.perseus-grc2:1400",
  "first_text_label": "Ἀγαμέμνων",
  "first_snippet": "ἥτις τοιόνδʼ ἐπʼ <em>ἀνδρὶ</em> κομπάζεις <em>λόγον</em>. ἥτις τοιόνδʼ ἐπʼ <em>ἀνδρὶ</em> κομπάζεις <em>λόγον</em>.",
  "author_scope": null
}


## Reader Search Within One Edition

`search_within_text` uses Scaife's reader search endpoint instead of the library-wide search endpoint. Use it when a prior discovery step has already selected an edition URN and you want local hits within that one text.


In [20]:
ILIAD_SCAIFE_EDITION = "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2"

async with Client(mcp) as client:
    within_text = await call_json(
        client,
        "search_within_text",
        {
            "query": "μῆνιν",
            "text_urn": ILIAD_SCAIFE_EDITION,
            "language": "greek",
            "query_format": "unicode",
            "search_kind": "form",
            "size": 5,
            "offset": 0,
        },
    )

assert within_text["total_count"] >= len(within_text.get("results", []))
print(json.dumps(summarize_search(within_text), ensure_ascii=False, indent=2))


{
  "total_count": 9,
  "page": null,
  "num_pages": null,
  "first_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1",
  "first_text_label": "Ἰλιάς",
  "first_snippet": "",
  "author_scope": null
}


## Passage-Level Highlights

`get_passage_highlights` asks Scaife for token-level matches in a single passage. This is useful for UI integrations, annotation workflows, or verifying exactly which token matched a search expression.


In [21]:
ILIAD_FIRST_LINE_SCAIFE = "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1"

async with Client(mcp) as client:
    highlights = await call_json(
        client,
        "get_passage_highlights",
        {
            "query": "μῆνιν",
            "passage_urn": ILIAD_FIRST_LINE_SCAIFE,
            "language": "greek",
            "query_format": "unicode",
        },
    )

assert highlights["total_count"] >= 1
assert "highlights" in highlights["results"][0]
print(json.dumps(highlights, ensure_ascii=False, indent=2)[:1500])


{
  "results": [
    {
      "passage": {
        "url": "/reader/urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1/",
        "json_url": "/library/passage/urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1/json/",
        "text_url": "/library/passage/urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1/text/",
        "text": {
          "url": "/library/urn:cts:greekLit:tlg0012.tlg001.perseus-grc2/",
          "json_url": "/library/urn:cts:greekLit:tlg0012.tlg001.perseus-grc2/json/",
          "text_url": "/library/passage/urn:cts:greekLit:tlg0012.tlg001.perseus-grc2/text/",
          "ancestors": [
            {
              "url": "/library/urn:cts:greekLit:tlg0012/",
              "json_url": "/library/urn:cts:greekLit:tlg0012/json/",
              "text_url": "/library/passage/urn:cts:greekLit:tlg0012/text/",
              "urn": "urn:cts:greekLit:tlg0012",
              "label": "Homer"
            },
            {
              "url": "/library/urn:cts:greekLit:tlg0012.tlg001/",
  

## Scaife Metadata and Text Retrieval

Scaife search results sometimes use edition URNs that do not match the CTS edition URNs exposed by Perseus. The Scaife-specific tools retrieve Scaife's own metadata and text directly, which provides a reliable fallback when CTS lookup fails or when you want to inspect the same resource Scaife returned.


In [22]:
async with Client(mcp) as client:
    library_metadata = await call_json(
        client,
        "get_scaife_library_metadata",
        {"urn": ILIAD_SCAIFE_EDITION},
    )
    passage_json = await call_json(
        client,
        "get_scaife_passage_json",
        {"urn": ILIAD_FIRST_LINE_SCAIFE},
    )
    passage_text = await call_text(
        client,
        "get_scaife_passage_text",
        {"urn": ILIAD_FIRST_LINE_SCAIFE},
    )

assert library_metadata["urn"] == ILIAD_SCAIFE_EDITION
assert passage_json["urn"] == ILIAD_FIRST_LINE_SCAIFE
assert "μῆνιν" in passage_text

print("Library label:", library_metadata.get("label"))
print("Passage URN:", passage_json.get("urn"))
print("Passage text:", passage_text.strip())


Library label: Ἰλιάς
Passage URN: urn:cts:greekLit:tlg0012.tlg001.perseus-grc2:1.1
Passage text: μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος


## What This Notebook Verifies

If all assertion cells pass, the advanced tool surface is working through the same MCP interface used by external clients. The notebook verifies registration, cache management, reference paging, server-side search scoping, author resolution, lemma/operator search, reader search, highlights, and Scaife metadata/text retrieval.
